## Full Local RAG Pipeline Using FAISS + FLAN-T5-base

**Step 1: Install Required Packages**

In [2]:
# Install necessary packages (only run once in your environment)
# pip install transformers sentencepiece

**Step 2: Import Required Libraries**

These libraries allow us to:

+ Load and search a FAISS vector index
+ Embed queries into vector space
+ Generate answers using the FLAN-T5-base language model

In [3]:
# Import necessary libraries 
# For fast vector similarity search
import faiss
# For handling CSV and DataFrame operations
import pandas as pd
# For encoding queries as embeddings
from sentence_transformers import SentenceTransformer
# For numerical operations
import numpy as np
# For text generation with FLAN-T5
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# Backend for transformers
import torch

c:\Users\brian\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Step 3: Load the FAISS Index and Chunked Text Data**

+ Loads the vector index previously created using PaddleOCR chunks.
+ Loads the original text chunks so that we can retrieve actual text (not just vectors) later.

In [4]:
# Load FAISS index and chunked text CSV
faiss_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\faiss_index_PaddleOCR.index"
csv_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_PaddleOCR2\cleaned_text_PaddleOCR2\chunked_text_PaddleOCR.csv"

# Load index and chunked data
index = faiss.read_index(faiss_path)
df_chunks = pd.read_csv(csv_path)

**Step 4: Load the SentenceTransformer Model**

We use the same model that was used for chunk embeddings to ensure semantic space alignment. This turns a query into a vector.

In [5]:
# Load SentenceTransformer model for query embedding
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

**Step 5: Define Function to Retrieve Top-k Chunks**

This function:

+ Encodes the question into an embedding.
+ Searches for the top k most similar document chunks in the FAISS index.
+ Returns the original chunks (not just IDs).

In [6]:
# === Step 5: Define function to retrieve top-k chunks ===
def search_top_k(query, k=1):  # use only top-1 to avoid overflow
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype("float32"), k)
    results = df_chunks.iloc[indices[0]]
    return results[["chunk_id", "filename", "chunk_text"]]

**Step 6: Load FLAN-T5-Base Model and Tokenizer**


+ FLAN-T5-Base is larger and more powerful than FLAN-T5-Small.
+ Tokenizer encodes and decodes input/output strings.
+ The model generates answers from the given prompts.

In [7]:
# Load FLAN-T5-Base model for answer generation 
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

**Step 7: Build the Prompt for the Model**

This function builds an instructional prompt using:
+ Context: Retrieved chunks
+ Question: The user's query
+ Answer: What the model will generate

In [8]:
# Build a clear and simple prompt with instruction ===
def build_prompt(chunks, question):
    context = "\n".join(chunks["chunk_text"].tolist())
    prompt = (
        "Use the information below to answer the question in one complete sentence.\n\n"
        "### Context ###\n"
        f"{context}\n\n"
        "### Question ###\n"
        f"{question}\n\n"
        "### Answer ###"
    )
    return prompt

**Step 8: Generate Answer from the LLM**

+ Tokenizes the prompt.
+ Generates a response using FLAN-T5.
+ Decodes and returns the final answer in plain English.

In [9]:
# Generate answer from the LLM 
def generate_answer(chunks, question, max_tokens=150):
    prompt = build_prompt(chunks, question)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = generation_model.generate(**inputs, max_new_tokens=max_tokens)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

**Step 9: Run a Sample Question**

+ You ask a question.
+ The most relevant document chunks are retrieved.
+ FLAN-T5 generates a response using that context.

In [10]:
# Run a sample question 
question = "What did Billy Joel's parents do in the 1930s?"
top_chunks = search_top_k(question)
response = generate_answer(top_chunks, question)

**Step 10: Display the Results**
This shows the answer generated by the model and the exact context used for that answer — useful for verification and debugging.

In [12]:
# Print the results
print("💬 Answer:", response)

💬 Answer:   Billy Joel 's mom dies at 92 ''


In [13]:
print("\n📄 Context used:")
print(top_chunks["chunk_text"].tolist())


📄 Context used:
["[ 14 . ^ a b c d e `` Billy Joel '' . Here 's The Thing . WNYC . July 30 , 2012 . Archived from the original transcript ) on August 4 , 2012.Retrieved August 3,2012 . `` I grew up on the Island , in the Levittown section of Hicksville.We had a Levitt house , Cape Cod , on the quarter acre . ... My father was a classically trained pianist . He grew up in Nuremberg , Germany , and he also went to school in Switzerland . His father was quite well off . They had a mail- order textile business , Joel Macht Fabrik ... '' 15 . ^ `` Billy Joel 's mom dies at 92 '' . Newsday . Archivedfrom the original on September 22 , 2020 . Retrieved October 3,2020 . 16.^Bego , Mark 2007 ) .Billy Joel : The Biography . Thunder 's Mouth Press . p.13 . ISBN 9781560259893 Charlie Rose.PBS.Archivedfrom the original on March 13 , 2020.Retrieved January 9,2020-via YouTube 18 . ^ a b Tallmer , Jerry ( July 22 , 2003 ) . `` Billy Joel grapples with the past '' . The Villager . Vol . 73 , no . 11 .